# 06 — OCR + multi-frame consensus + сборка submission CSV + оценка по GT

**Что делает:**
1. Для каждого трека берёт **top-K=10 кропов** из `data/best_crops/<video>/<tid>_<rank>.jpg`
2. На каждом — PaddleOCR + WeChat QR + zxing (с кешем per crop_path)
3. **Консенсус:**
   - **barcode**: предпочитаем zxing-результат с валидной EAN-13 контрольной суммой, иначе первый найденный
   - **QR**: первый непустой dict
   - **текстовые поля** (`parse_all`): majority vote по каждому полю через K кропов
   - **product_name**: majority vote через K
4. Пишет `data/submission.csv` со всеми 25 полями
5. Сопоставляет с GT по IoU≥0.5 → strict accuracy + soft Jaccard + plausibility

**Вход:** `data/best_crops/`, `data/best_crops_meta.json` (с полем `crops: [...]`), `data/per_frame_tracks.json`, `data/crops_meta.json`  
**Выход:** `data/submission.csv`, `data/eval_report.json`, `data/ocr_cache.json`

## 1. Импорты и загрузка парсеров из 05_parsers.ipynb

In [33]:
import json
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import nbformat

ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
BEST_CROPS_DIR  = ROOT / 'data' / 'best_crops'
BEST_META       = ROOT / 'data' / 'best_crops_meta.json'
PER_FRAME       = ROOT / 'data' / 'per_frame_tracks.json'
GT_META         = ROOT / 'data' / 'crops_meta.json'
SUBMISSION_CSV  = ROOT / 'data' / 'submission.csv'
EVAL_REPORT     = ROOT / 'data' / 'eval_report.json'

# Загружаем все определения из 05_parsers.ipynb
parsers_nb = nbformat.read(ROOT / 'eda' / '05_parsers.ipynb', as_version=4)
for cell in parsers_nb.cells:
    if cell.cell_type == 'code':
        exec(cell.source, globals())

print('Парсеры загружены из 05_parsers.ipynb')
print('Доступны:', [n for n in ('parse_all', 'parse_qr_payload', 'OcrLine') if n in globals()])

barcode: 4670025474665
checksum OK: True
id_sku: 270207736530
datetime full : 03.04.2026 3:08
datetime fixed: 03.04.2026 3:08
datetime split: 03.04.2026 3:08
discount: -48%
prices full : {'price_default': None, 'price_card': '129,99'}
prices fallback (int-only default): {'price_default': '252,00', 'price_card': '129,99'}
exact : Сухое
OCR-err Полусухое→Палусухое: Полусухое
OCR-err Сухое→Сyое: None
OCR-err Сухое→Cyxoe (Latin): Сухое
OCR-err Полусладкое→Полусладко: Сладкое
no hint: None
K в кропе : К
Ш в кропе : Ш
латин. K  : К
в слове   : None
пусто     : None
Парсеры загружены из 05_parsers.ipynb
Доступны: ['parse_all', 'parse_qr_payload', 'OcrLine']


## 2. Инициализация PaddleOCR + WeChat QR

In [34]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang='ru', use_textline_orientation=True)
print('PaddleOCR готов')

try:
    qr_reader = cv2.wechat_qrcode_WeChatQRCode()
    print('WeChat QR готов')
except Exception as e:
    qr_reader = None
    print(f'WeChat QR недоступен: {e}')

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/alex/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/alex/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/alex/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/alex/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('eslav_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Us

PaddleOCR готов
WeChat QR готов


In [35]:
import os
import json
import time
import cv2

from collections import defaultdict

VIDEO_MAP = {
    '25_12-20': ROOT / 'Данные/25_12-20/25_12-20.mp4',
    '26_12-20': ROOT / 'Данные/26_12-20/26_12-20.mp4',
    '43_15': ROOT / 'Данные/43_15/43_15.mp4',
    'Unlabeled_25_12-20': ROOT / 'Данные/Unlabeled/25_12-20.mp4',
    'Unlabeled_26_12-20': ROOT / 'Данные/Unlabeled/26_12-20.mp4',
    'Unlabeled_26_2-10': ROOT / 'Данные/Unlabeled/26_2-10.mp4',
}

meta_all = json.loads(BEST_META.read_text(encoding='utf-8'))

by_video = defaultdict(list)

for m in meta_all:
    by_video[m['video']].append(m)

# Сортируем все кропы по frame_idx — это сводит seek-расходы
for vid in by_video:
    by_video[vid].sort(key=lambda r: r['best_frame_idx'])

t0 = time.time()
total = 0

for vid, tracks in by_video.items():
    cap = cv2.VideoCapture(str(VIDEO_MAP[vid]))

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Собираем плоский список (frame_idx, bbox, crop_path) и сортируем по frame_idx
    flat = []

    for m in tracks:
        for c in m['crops']:
            flat.append((c['frame_idx'], c['bbox'], c['crop_path']))

    flat.sort(key=lambda r: r[0])

    for frame_idx, bbox, crop_path in flat:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

        ok, frame = cap.read()

        if not ok:
            continue

        x1, y1, x2, y2 = bbox

        x1p = max(0, int(x1))
        y1p = max(0, int(y1))
        x2p = min(W, int(x2))
        y2p = min(H, int(y2))

        crop = frame[y1p:y2p, x1p:x2p]

        if crop.size == 0:
            continue

        cv2.imwrite(
            str(ROOT / crop_path),
            crop,
            [cv2.IMWRITE_JPEG_QUALITY, 95],
        )

        total += 1

    cap.release()

    print(f'  {vid}: {len(tracks)} треков, {len(flat)} кропов')

print(f'\nГотово: {total} кропов за {(time.time() - t0) / 60:.1f} мин')

sample = meta_all[0]
sz = os.path.getsize(ROOT / sample['crop_path'])
img = cv2.imread(str(ROOT / sample['crop_path']))

print(f'Пример (best): {sample["crop_path"]}  {img.shape}  ({sz // 1024} KB)')
print(f'  Всего кропов в этом треке: {len(sample["crops"])}')

  25_12-20: 109 треков, 909 кропов
  26_12-20: 153 треков, 1402 кропов
  43_15: 41 треков, 379 кропов
  Unlabeled_25_12-20: 86 треков, 757 кропов
  Unlabeled_26_12-20: 88 треков, 757 кропов
  Unlabeled_26_2-10: 67 треков, 574 кропов

Готово: 4766 кропов за 19.9 мин
Пример (best): data/best_crops/25_12-20/0001_0.jpg  (278, 272, 3)  (47 KB)
  Всего кропов в этом треке: 10


## 3. Helper: один кроп → OcrLine[] + QR payload

In [43]:
def ocr_lines(img: np.ndarray) -> list:
    """PaddleOCR на BGR-изображении → список OcrLine."""
    if img is None or img.size == 0:
        return []
    res = ocr.predict(img)
    if not res or not res[0]:
        return []
    out = []

    r = res[0]
    if isinstance(r, dict):
        texts = r.get('rec_texts', [])
        scores = r.get('rec_scores', [])
        boxes = r.get('rec_boxes', r.get('rec_polys', []))
        for t, s, b in zip(texts, scores, boxes):
            arr = np.array(b).reshape(-1, 2)
            x = float(arr[:, 0].min()); y = float(arr[:, 1].min())
            w = float(arr[:, 0].max() - x); h = float(arr[:, 1].max() - y)
            out.append(OcrLine(t, x, y, w, h, float(s)))
    else:
        
        for item in r:
            box, (text, score) = item[0], item[1]
            arr = np.array(box)
            x = float(arr[:, 0].min()); y = float(arr[:, 1].min())
            w = float(arr[:, 0].max() - x); h = float(arr[:, 1].max() - y)
            out.append(OcrLine(text, x, y, w, h, float(score)))
    return out


def decode_qr(img: np.ndarray) -> dict:
    """WeChat QR декодер. Возвращает dict с QR-полями или пустой dict."""
    if qr_reader is None or img is None or img.size == 0:
        return {}
    try:
        texts, _ = qr_reader.detectAndDecode(img)
    except Exception:
        return {}
    for t in texts or []:
        if t:
            parsed = parse_qr_payload(t)
            if parsed:
                return parsed
    return {}


import json as _json
_meta = json.loads(BEST_META.read_text())
_sample = _meta[0]
_img = cv2.imread(str(ROOT / _sample['crop_path']))
_lines = ocr_lines(_img)
_qr = decode_qr(_img)
print(f"Кроп: {_sample['crop_path']}")
print(f"  OCR строк: {len(_lines)}")
for ln in _lines[:5]:
    print(f"    {ln.text!r}  conf={ln.conf:.2f}")
print(f"  QR полей: {len(_qr)} → {_qr}")
print(f"\n  parse_all → {parse_all(_lines)}")

Кроп: data/best_crops/25_12-20/0001_0.jpg
  OCR строк: 12
    'Напиток'  conf=0.78
    'безалкогальный'  conf=0.87
    'ABRAU LGHT'  conf=0.95
    'Apomar. c raa.'  conf=0.82
    '(РОCСHя) 0. 25L'  conf=0.66
  QR полей: 0 → {}

  parse_all → {'barcode': '4680140271438', 'id_sku': '770207716035', 'print_datetime': None, 'discount_amount': '-10%', 'additional_info': None, 'special_symbols': None, 'price_default': None, 'price_card': '129,99'}


In [52]:
import re
import zxingcpp
from collections import Counter

OCR_CACHE = ROOT / 'data' / 'ocr_cache.json'

# ── Альтернативные QR-декодеры ──────────────────────────────────────────
# WeChat пропустил 542/544 ценников. cv2 QR на исходном размере тоже не справился
# (QR ~80×80 px ≈ 2.7 px/модуль — ниже порога). Пробуем с апскейлом ×3.

_cv2_qr = cv2.QRCodeDetector()

try:
    from pyzbar import pyzbar as _pyzbar
    _HAS_PYZBAR = True
    print('pyzbar доступен — будет использован для QR fallback')
except Exception as _e:
    _HAS_PYZBAR = False
    print(f'pyzbar недоступен ({_e!r}) — используем только OpenCV QR fallback')


_QR_UPSCALES = (1, 2, 3, 4)   # пробуем разные масштабы


def _try_decoders(img):
    """OpenCV → pyzbar (если есть). Возвращает parsed dict или {}."""
    try:
        data, points, _ = _cv2_qr.detectAndDecode(img)
        if data:
            p = parse_qr_payload(data)
            if p:
                return p
    except Exception:
        pass

    if _HAS_PYZBAR:
        try:
            for d in _pyzbar.decode(img):
                if str(d.type).upper() == 'QRCODE':
                    t = d.data.decode('utf-8', errors='ignore')
                    p = parse_qr_payload(t)
                    if p:
                        return p
        except Exception:
            pass

    return {}


def decode_qr_alt(img):
    """Пробуем альтернативные декодеры на разных масштабах кропа.
    Малые QR на ценнике (~80px) нужно апскейлить, чтобы достичь ≥4 px/модуль."""
    if img is None or img.size == 0:
        return {}

    h, w = img.shape[:2]

    for scale in _QR_UPSCALES:
        if scale == 1:
            tgt = img
        else:
            tgt = cv2.resize(img, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
        result = _try_decoders(tgt)
        if result:
            return result

    return {}


def decode_barcode_1d(img):
    """zxing-cpp на BGR-кропе → строка 1D-штрихкода или None."""
    if img is None or img.size == 0:
        return None
    try:
        results = zxingcpp.read_barcodes(img)
    except Exception:
        return None
    for r in results:
        fmt = str(r.format).split('.')[-1]
        if fmt in ('EAN13', 'EAN8', 'UPCA', 'UPCE', 'Code128', 'Code39', 'ITF'):
            return r.text
    return None


def line_to_dict(L):
    return {'text': L.text, 'x': L.x, 'y': L.y, 'w': L.w, 'h': L.h, 'conf': L.conf}

def dict_to_line(d):
    return OcrLine(d['text'], d['x'], d['y'], d['w'], d['h'], d['conf'])


def build_product_name(lines):
    if not lines: return None
    max_y = max(L.y + L.h for L in lines)
    cutoff = max_y * 0.55
    top = [L for L in lines if (L.y + L.h / 2) < cutoff
                            and any(c.isalpha() for c in L.text)
                            and len(L.text.strip()) >= 2]
    if not top: return None
    top.sort(key=lambda L: (L.y, L.x))
    name = ' '.join(L.text.strip() for L in top)
    return name if len(name) >= 5 else None


# ── Валидаторы ──

def is_valid_ean13(s):
    if not s: return False
    digits = ''.join(c for c in str(s) if c.isdigit())
    return len(digits) == 13 and ean13_checksum_ok(digits)

def valid_barcode(s):
    if not s: return False
    d = ''.join(c for c in str(s) if c.isdigit())
    if len(d) == 13:
        return ean13_checksum_ok(d)
    return len(d) in (8, 12)

def valid_id_sku(s):
    if not s: return False
    d = ''.join(c for c in str(s) if c.isdigit())
    return 9 <= len(d) <= 12

def valid_price(s):
    if not s: return False
    try:
        f = float(str(s).replace(',', '.'))
        return 10.0 <= f <= 99999.0
    except Exception:
        return False

def valid_datetime(s):
    if not s: return False
    return bool(re.fullmatch(r'\d{2}\.\d{2}\.\d{4}( \d{1,2}:\d{2})?', str(s)))

def valid_discount(s):
    if not s: return False
    m = re.fullmatch(r'-\d{1,2}%', str(s))
    if not m: return False
    return 1 <= int(str(s)[1:-1]) <= 99

def valid_special_sym(s):
    return s in ('К', 'Ш')

VALIDATORS = {
    'barcode':         valid_barcode,
    'id_sku':          valid_id_sku,
    'price_default':   valid_price,
    'price_card':      valid_price,
    'print_datetime':  valid_datetime,
    'discount_amount': valid_discount,
    'special_symbols': valid_special_sym,
}


ocr_cache = {}
if OCR_CACHE.exists():
    ocr_cache = json.loads(OCR_CACHE.read_text())
    # Инвалидируем пустые qr_alt — они получены без апскейла и больше не валидны
    n_invalidated = 0
    for k, cd in ocr_cache.items():
        if 'qr_alt' in cd and not cd['qr_alt']:
            del cd['qr_alt']
            n_invalidated += 1
    print(f'Загружен OCR-кэш: {len(ocr_cache)} кропов, инвалидирован пустой qr_alt у {n_invalidated}')
else:
    print('OCR-кэш не найден, будет создан при первом прогоне')


CSV_FIELDS = [
    'video', 'track_id',
    'product_name', 'price_default', 'price_card', 'price_discount',
    'barcode', 'discount_amount', 'id_sku', 'print_datetime',
    'code', 'additional_info', 'color', 'special_symbols',
    'qr_code_barcode', 'price1_qr', 'price2_qr', 'price3_qr', 'price4_qr',
    'wholesale_level_1_count', 'wholesale_level_1_price',
    'wholesale_level_2_count', 'wholesale_level_2_price',
    'action_price_qr', 'action_code_qr',
]

LIMIT_FIRST_N = None
FORCE_REOCR   = False

meta = json.loads(BEST_META.read_text())
if LIMIT_FIRST_N: meta = meta[:LIMIT_FIRST_N]

records = []
n_zxing_hit = n_qr_hit = n_qr_alt_hit = n_cache_hit = n_total_crops = 0
n_dropped = Counter()
n_save_every = 50
t0 = time.time()

for i, m in enumerate(meta, 1):
    crop_results = []

    for c in m['crops']:
        cache_key = c['crop_path']
        n_total_crops += 1
        img_loaded = None

        if not FORCE_REOCR and cache_key in ocr_cache:
            cd = ocr_cache[cache_key]
            lines = [dict_to_line(d) for d in cd['lines']]
            qr = cd['qr']
            zx = cd['zxing']
            n_cache_hit += 1
        else:
            img_loaded = cv2.imread(str(ROOT / c['crop_path']))
            lines = ocr_lines(img_loaded)
            qr = decode_qr(img_loaded)
            zx = decode_barcode_1d(img_loaded)
            ocr_cache[cache_key] = {
                'lines': [line_to_dict(L) for L in lines],
                'qr': qr,
                'zxing': zx,
            }

        # ── QR alt-decoder с апскейлом: если WeChat не нашёл, пробуем cv2/pyzbar на ×2,3,4 ──
        if not qr:
            cd = ocr_cache.get(cache_key, {})
            if 'qr_alt' in cd:
                if cd['qr_alt']:
                    qr = cd['qr_alt']
            else:
                if img_loaded is None:
                    img_loaded = cv2.imread(str(ROOT / c['crop_path']))
                qr_alt = decode_qr_alt(img_loaded)
                cd['qr_alt'] = qr_alt
                ocr_cache[cache_key] = cd
                if qr_alt:
                    qr = qr_alt
                    n_qr_alt_hit += 1

        crop_results.append({'lines': lines, 'qr': qr, 'zxing': zx})

    # ── Консенсус ──

    zxing_barcode = next((c['zxing'] for c in crop_results
                          if c['zxing'] and is_valid_ean13(c['zxing'])), None)

    qr = next((c['qr'] for c in crop_results if c['qr']), {})

    field_votes = defaultdict(Counter)
    for c in crop_results:
        if not c['lines']: continue
        parsed_c = parse_all(c['lines'])
        for k, v in (parsed_c or {}).items():
            if v:
                field_votes[k][v] += 1
    parsed = {k: votes.most_common(1)[0][0] for k, votes in field_votes.items()}

    name_votes = Counter()
    for c in crop_results:
        pn = build_product_name(c['lines'])
        if pn: name_votes[pn] += 1
    product_name = name_votes.most_common(1)[0][0] if name_votes else None

    if zxing_barcode: n_zxing_hit += 1
    if qr: n_qr_hit += 1

    # ── Запись с валидацией ──
    rec = {f: 'нет' for f in CSV_FIELDS}
    rec['video']           = m['video']
    rec['track_id']        = m['track_id']
    rec['color']           = 'red'
    rec['price_discount']  = 'нет'
    rec['code']            = 'нет'
    rec['special_symbols'] = 'нет'

    for k in ('barcode', 'id_sku', 'print_datetime', 'discount_amount',
              'additional_info', 'special_symbols',
              'price_card', 'price_default'):
        v = parsed.get(k)
        if not v:
            continue
        validator = VALIDATORS.get(k)
        if validator and not validator(v):
            n_dropped[k] += 1
            continue
        rec[k] = v

    if zxing_barcode:
        rec['barcode'] = zxing_barcode

    for k, v in qr.items():
        if k in rec and v is not None:
            rec[k] = v

    if product_name:
        rec['product_name'] = product_name

    records.append(rec)

    if i % n_save_every == 0:
        rate = i / (time.time() - t0)
        eta = (len(meta) - i) / rate
        print(f'  {i}/{len(meta)} треков, {n_total_crops} кропов '
              f'({rate:.1f} tr/s, ETA {eta/60:.1f} мин, '
              f'cache={n_cache_hit}/{n_total_crops}, '
              f'zxing={n_zxing_hit}, qr={n_qr_hit}, qr_alt-only={n_qr_alt_hit})')
        OCR_CACHE.write_text(json.dumps(ocr_cache, ensure_ascii=False))

OCR_CACHE.write_text(json.dumps(ocr_cache, ensure_ascii=False))
print(f'\nГотово за {(time.time()-t0)/60:.1f} мин, треков: {len(records)}, кропов: {n_total_crops}')
print(f'  cache hits:        {n_cache_hit}/{n_total_crops} ({n_cache_hit/max(n_total_crops,1):.0%})')
print(f'  zxing valid EAN13: {n_zxing_hit}/{len(records)} треков')
print(f'  QR (any decoder):  {n_qr_hit}/{len(records)} треков')
print(f'    из них alt-only: {n_qr_alt_hit} кропов отдали QR через cv2 с апскейлом')
if n_dropped:
    print(f'  Отвергнуто валидаторами:')
    for f, n in n_dropped.most_common():
        print(f'    {f:<18} {n}')

pyzbar недоступен (ImportError('Unable to find zbar shared library')) — используем только OpenCV QR fallback
Загружен OCR-кэш: 5322 кропов, инвалидирован пустой qr_alt у 4758
  50/544 треков, 445 кропов (2.2 tr/s, ETA 3.8 мин, cache=445/445, zxing=0, qr=0, qr_alt-only=0)
  100/544 треков, 835 кропов (2.2 tr/s, ETA 3.3 мин, cache=835/835, zxing=0, qr=0, qr_alt-only=0)
  150/544 треков, 1282 кропов (2.2 tr/s, ETA 3.0 мин, cache=1282/1282, zxing=0, qr=2, qr_alt-only=0)
  200/544 треков, 1730 кропов (2.2 tr/s, ETA 2.7 мин, cache=1730/1730, zxing=0, qr=2, qr_alt-only=0)
  250/544 треков, 2198 кропов (1.9 tr/s, ETA 2.5 мин, cache=2198/2198, zxing=0, qr=2, qr_alt-only=0)
  300/544 треков, 2660 кропов (2.0 tr/s, ETA 2.0 мин, cache=2660/2660, zxing=0, qr=2, qr_alt-only=0)
  350/544 треков, 3117 кропов (2.1 tr/s, ETA 1.5 мин, cache=3117/3117, zxing=0, qr=2, qr_alt-only=0)
  400/544 треков, 3557 кропов (2.2 tr/s, ETA 1.1 мин, cache=3557/3557, zxing=0, qr=2, qr_alt-only=0)
  450/544 треков, 3959 к

## 5. Submission CSV

In [ ]:
# Формат submission под sample.csv:
#   - разделитель ','
#   - 29 колонок (filename, ..., frame_timestamp, x_min, y_min, x_max, y_max, ..., qr)
#   - все цены в DOT-decimal (1234.56), включая обычные (у нас внутри они comma-decimal)
#   - missing = 'нет'
#
# Внутренний `records` НЕ меняем — он используется eval-ячейкой и должен быть согласован с GT
# (где обычные цены в COMMA-decimal: '129,99').

SUBMISSION_FIELDS = [
    'filename',
    'product_name', 'price_default', 'price_card', 'price_discount',
    'barcode', 'discount_amount', 'id_sku', 'print_datetime', 'code',
    'additional_info', 'color', 'special_symbols',
    'frame_timestamp', 'x_min', 'y_min', 'x_max', 'y_max',
    'qr_code_barcode', 'price1_qr', 'price2_qr', 'price3_qr', 'price4_qr',
    'wholesale_level_1_count', 'wholesale_level_1_price',
    'wholesale_level_2_count', 'wholesale_level_2_price',
    'action_price_qr', 'action_code_qr',
]

# Поля, в которых надо заменить ',' → '.' (обычные цены, внутри они comma-decimal).
# QR-цены уже в dot-decimal (приходят из parse_qr_payload напрямую из URL-encoded payload).
COMMA_TO_DOT_FIELDS = {'price_default', 'price_card', 'price_discount'}


def comma_to_dot(v):
    """'1234,56' → '1234.56'. Не трогаем 'нет' и нечисловые."""
    if v in (None, '', 'нет'):
        return v
    s = str(v)
    # Только если выглядит как число с запятой
    if re.fullmatch(r'-?\d+,\d+', s):
        return s.replace(',', '.')
    return s


# Индексируем meta_all для быстрого поиска (video, track_id) → meta
meta_lookup = {(m['video'], m['track_id']): m for m in meta_all}

# filename из video_id: '25_12-20' → '25_12-20.mp4', 'Unlabeled_26_2-10' → '26_2-10.mp4'
def video_to_filename(video_id):
    return VIDEO_MAP[video_id].name


export_rows = []

for r in records:
    m = meta_lookup.get((r['video'], r['track_id']))
    bbox = m['bbox'] if m else [0, 0, 0, 0]
    ts_ms = int(round(m['best_timestamp_ms'])) if m else 0

    row = {
        'filename':        video_to_filename(r['video']),
        'frame_timestamp': ts_ms,
        'x_min':           round(float(bbox[0]), 1),
        'y_min':           round(float(bbox[1]), 1),
        'x_max':           round(float(bbox[2]), 1),
        'y_max':           round(float(bbox[3]), 1),
    }

    for f in SUBMISSION_FIELDS:
        if f in row:  # уже заполнено
            continue
        v = r.get(f, 'нет')
        if f in COMMA_TO_DOT_FIELDS:
            v = comma_to_dot(v)
        row[f] = v

    export_rows.append(row)


df = pd.DataFrame(export_rows, columns=SUBMISSION_FIELDS)
df.to_csv(SUBMISSION_CSV, sep=',', index=False, encoding='utf-8')

print(f'Записан {SUBMISSION_CSV}  ({SUBMISSION_CSV.stat().st_size//1024} KB, {len(df)} строк, {len(df.columns)} колонок)')
print(f'\nПервая строка:')
print(df.iloc[0].to_dict())
df.head(3)

## 6. Оценка

In [54]:
def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0: return 0.0
    return inter / ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter)

gt = json.loads(GT_META.read_text())
per_frame = json.loads(PER_FRAME.read_text())
fps_map = {'25_12-20': 19.6, '26_12-20': 20.0, '43_15': 19.9}
labeled_videos = set(fps_map)

pred_lookup = {(r['video'], r['track_id']): r for r in records}

matches = []  # list of (gt_record, predicted_record)
for g in gt:
    vid = g['video']
    if vid not in labeled_videos: continue
    gt_frame = int(round(g['frame_timestamp_ms'] / 1000.0 * fps_map[vid]))
    frame_map = per_frame.get(vid, {})
    best_iou, best_tid = 0.0, None
    for delta in range(-2, 3):
        for t in frame_map.get(str(gt_frame + delta), []):
            i = iou(g['bbox'], t['bbox'])
            if i > best_iou:
                best_iou, best_tid = i, t['track_id']
    if best_iou >= 0.5 and (vid, best_tid) in pred_lookup:
        matches.append((g, pred_lookup[(vid, best_tid)]))

print(f'Совпало: {len(matches)} GT ↔ predicted')

Совпало: 124 GT ↔ predicted


In [ ]:
PER_TAG_THRESHOLD = 0.80
EVAL_FIELDS = [f for f in CSV_FIELDS if f not in ('video', 'track_id')]
BASE_CORRECT_IF_MATCHED = 6
TOTAL_FIELDS_PER_TAG = len(EVAL_FIELDS) + BASE_CORRECT_IF_MATCHED

def norm(v):
    if v is None: return 'нет'
    s = str(v).strip()
    return s if s else 'нет'


pred_by_match = {id(g): p for g, p in matches}

tag_scores = []
field_stats = {f: {'correct': 0, 'total': 0, 'examples_wrong': []} for f in EVAL_FIELDS}

for g in gt:
    vid = g['video']
    if vid not in labeled_videos: continue
    matched = id(g) in pred_by_match
    if matched:
        p = pred_by_match[id(g)]
        n_correct = BASE_CORRECT_IF_MATCHED   # bbox+filename+timestamp
        for f in EVAL_FIELDS:
            gt_v = norm(g.get(f, 'нет'))
            pr_v = norm(p.get(f, 'нет'))
            field_stats[f]['total'] += 1
            if gt_v == pr_v:
                n_correct += 1
                field_stats[f]['correct'] += 1
            elif len(field_stats[f]['examples_wrong']) < 3:
                field_stats[f]['examples_wrong'].append({'gt': gt_v, 'pred': pr_v})
    else:
        n_correct = 0
    tag_scores.append({'video': vid, 'correct': n_correct,
                       'pct': n_correct / TOTAL_FIELDS_PER_TAG, 'matched': matched})


passed = sum(1 for t in tag_scores if t['pct'] >= PER_TAG_THRESHOLD)
total  = len(tag_scores)
print(f'  Ценников с точностью ≥ {PER_TAG_THRESHOLD:.0%}: {passed}/{total} = {passed/total:.1%}')

# Распределение
print('\nРаспределение per-tag accuracy (всего 157):')
buckets = [(0.0, 0.01), (0.01, 0.5), (0.5, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
for lo, hi in buckets:
    n = sum(1 for t in tag_scores if lo <= t['pct'] < hi)
    bar = '█' * int(40 * n / total)
    pct_label = '0% (миссы)' if lo == 0 else f'{lo:.0%}-{hi:.0%}'
    print(f'  {pct_label:>13}: {n:3d} {bar}')

# Per-video
print('\nПо видео:')
for vid in sorted(labeled_videos):
    vs = [t for t in tag_scores if t['video'] == vid]
    pas = sum(1 for t in vs if t['pct'] >= PER_TAG_THRESHOLD)
    matched = sum(1 for t in vs if t['matched'])
    print(f'  {vid}: passed {pas}/{len(vs)} ({pas/len(vs):.0%}), capture {matched}/{len(vs)} ({matched/len(vs):.0%})')

# Per-field (для дебага)
print(f'\nPer-field accuracy на {len(matches)} matched (для отладки):')
rows = sorted(((f, s['correct'], s['total']) for f, s in field_stats.items()),
              key=lambda r: r[1]/r[2] if r[2] else 0, reverse=True)
for f, c, t in rows:
    print(f'  {f:<26} {c:>3}/{t:<3} = {c/t:.1%}' if t else f'  {f:<26}  (нет матчей)')

EVAL_REPORT.write_text(json.dumps({
    'main_metric_pct': passed / total,
    'passed_tags':  passed,
    'total_tags':   total,
    'matched_tags': sum(1 for t in tag_scores if t['matched']),
    'capture_rate': sum(1 for t in tag_scores if t['matched']) / total,
    'per_video':    {vid: {
        'passed':  sum(1 for t in tag_scores if t['video']==vid and t['pct']>=PER_TAG_THRESHOLD),
        'matched': sum(1 for t in tag_scores if t['video']==vid and t['matched']),
        'total':   sum(1 for t in tag_scores if t['video']==vid)} for vid in labeled_videos},
    'per_field':    field_stats,
}, ensure_ascii=False, indent=1))
print(f'\nReport saved: {EVAL_REPORT}')

  Ценников с точностью ≥ 80%: 1/157 = 0.6%

Распределение per-tag accuracy (всего 157):
     0% (миссы):  33 ████████
         1%-50%:   0 
        50%-70%: 113 ████████████████████████████
        70%-80%:  10 ██
        80%-90%:   1 
       90%-101%:   0 

По видео:
  25_12-20: passed 0/57 (0%), capture 54/57 (95%)
  26_12-20: passed 1/71 (1%), capture 42/71 (59%)
  43_15: passed 0/29 (0%), capture 28/29 (97%)

Per-field accuracy на 124 matched (для отладки):
  price_discount             124/124 = 100.0%
  price3_qr                  124/124 = 100.0%
  action_price_qr            124/124 = 100.0%
  action_code_qr             124/124 = 100.0%
  color                      123/124 = 99.2%
  wholesale_level_1_count    123/124 = 99.2%
  wholesale_level_1_price    123/124 = 99.2%
  wholesale_level_2_count    123/124 = 99.2%
  wholesale_level_2_price    123/124 = 99.2%
  discount_amount             84/124 = 67.7%
  price_card                  79/124 = 63.7%
  additional_info             76/12

In [55]:
# Диагностика: ценники в bucket 70-80% — что им конкретно не хватает?
# Это те, кому до прохождения главной метрики (≥80%) нужно ещё 1-3 поля.

near_pass = []  # (g, p, missing_fields)

for g, p in matches:
    n_correct = BASE_CORRECT_IF_MATCHED
    wrong = []
    for f in EVAL_FIELDS:
        gt_v = norm(g.get(f, 'нет'))
        pr_v = norm(p.get(f, 'нет'))
        if gt_v == pr_v:
            n_correct += 1
        else:
            wrong.append((f, gt_v, pr_v))
    pct = n_correct / TOTAL_FIELDS_PER_TAG
    if 0.70 <= pct < 0.80:
        near_pass.append((g, p, pct, wrong))

near_pass.sort(key=lambda r: -r[2])

print(f'Ценников в bucket 70-80% (нужно ещё 1-3 поля до прохождения): {len(near_pass)}\n')

# Какие поля чаще всего «не хватает» этим ценникам?
miss_counter = Counter()
miss_by_pair = Counter()  # (поле, gt→pred) — частые конкретные ошибки

for g, p, pct, wrong in near_pass:
    for f, gt_v, pr_v in wrong:
        miss_counter[f] += 1
        # Сокращаем строки для счётчика
        gt_short = gt_v[:30] + ('…' if len(gt_v) > 30 else '')
        pr_short = pr_v[:30] + ('…' if len(pr_v) > 30 else '')
        miss_by_pair[(f, gt_short, pr_short)] += 1

print('Поля, в которых ошибается этот bucket (топ):')
for f, n in miss_counter.most_common():
    print(f'  {f:<22} {n:>3}/{len(near_pass)} ценников')

print('\nЧастые конкретные расхождения (топ-10):')
for (f, gt_v, pr_v), n in miss_by_pair.most_common(10):
    print(f'  [{n}x] {f}: GT={gt_v!r}  PRED={pr_v!r}')

print(f'\nДетали по 3 ближайшим к порогу:')
for g, p, pct, wrong in near_pass[:3]:
    print(f'\n  {p["video"]}/track {p["track_id"]} — pct={pct:.0%}, '
          f'верно {int(pct * TOTAL_FIELDS_PER_TAG)}/{TOTAL_FIELDS_PER_TAG}, '
          f'не хватает {TOTAL_FIELDS_PER_TAG - int(pct * TOTAL_FIELDS_PER_TAG)} поля до 80%')
    for f, gt_v, pr_v in wrong:
        print(f'      {f:<22} GT={gt_v!r:<30} PRED={pr_v!r}')

Ценников в bucket 70-80% (нужно ещё 1-3 поля до прохождения): 10

Поля, в которых ошибается этот bucket (топ):
  product_name            10/10 ценников
  price_default           10/10 ценников
  qr_code_barcode          8/10 ценников
  price1_qr                8/10 ценников
  price4_qr                8/10 ценников
  price2_qr                7/10 ценников
  barcode                  6/10 ценников
  id_sku                   5/10 ценников
  additional_info          3/10 ценников
  special_symbols          3/10 ценников
  print_datetime           2/10 ценников
  code                     2/10 ценников
  price_card               2/10 ценников
  discount_amount          1/10 ценников

Частые конкретные расхождения (топ-10):
  [2x] price1_qr: GT='1368.42'  PRED='нет'
  [2x] price2_qr: GT='1299.99'  PRED='нет'
  [2x] special_symbols: GT='К'  PRED='нет'
  [2x] price4_qr: GT='1299.99'  PRED='нет'
  [2x] price2_qr: GT='1999.99'  PRED='нет'
  [1x] product_name: GT='Вино CHATEAU LE VIEUX FORT Бор…'  

In [50]:
import re as _re

# ── Plausibility-чекеры (выглядит ли значение разумно) ──
def _plausible_barcode(v):
    if v in (None, 'нет'): return None  # пропуск — нет значения
    digits = _re.sub(r'\D', '', str(v))
    if len(digits) == 13:
        return ean13_checksum_ok(digits)
    return len(digits) in (8, 12)

def _plausible_id_sku(v):
    if v in (None, 'нет'): return None
    digits = _re.sub(r'\D', '', str(v))
    return 9 <= len(digits) <= 12

def _plausible_price(v):
    if v in (None, 'нет'): return None
    s = str(v).replace(',', '.')
    try:
        f = float(s)
        return 10.0 <= f <= 99999.0
    except Exception:
        return False

def _plausible_price_card_99(v):
    return _plausible_price(v) and str(v).endswith(',99')

def _plausible_discount(v):
    if v in (None, 'нет'): return None
    m = _re.fullmatch(r'-\d{1,2}%', str(v))
    if not m: return False
    n = int(str(v)[1:-1])
    return 1 <= n <= 99

def _plausible_datetime(v):
    if v in (None, 'нет'): return None
    return bool(_re.fullmatch(r'\d{2}\.\d{2}\.\d{4}( \d{1,2}:\d{2})?', str(v)))

def _plausible_color(v):
    return v in ('red', 'yellow') if v not in (None, 'нет') else None

def _plausible_additional(v):
    return v in ADDITIONAL_HINTS if v not in (None, 'нет') else None

def _plausible_special_sym(v):
    return v in ('К', 'Ш') if v not in (None, 'нет') else None

PLAUSIBLE = {
    'barcode': _plausible_barcode,
    'id_sku': _plausible_id_sku,
    'price_default': _plausible_price,
    'price_card': _plausible_price_card_99,
    'price_discount': _plausible_price,
    'discount_amount': _plausible_discount,
    'print_datetime': _plausible_datetime,
    'color': _plausible_color,
    'additional_info': _plausible_additional,
    'special_symbols': _plausible_special_sym,
    'qr_code_barcode': _plausible_barcode,
}

# ── Soft similarity (char-level Jaccard) ──
def char_jaccard(a, b):
    """|set(a) ∩ set(b)| / |set(a) ∪ set(b)| на нормализованных строках.
    Для пары ('нет','нет') возвращает 1.0; для одной 'нет' — 0.0."""
    a, b = norm(a), norm(b)
    if a == b: return 1.0
    sa, sb = set(a.lower()), set(b.lower())
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

# ── Считаем все три метрики ──
NON_EMPTY = lambda v: v not in (None, '', 'нет')

# Coverage: процент записей с НЕ-`нет` значением
coverage = {}
for f in EVAL_FIELDS:
    n_filled = sum(1 for r in records if NON_EMPTY(r.get(f)))
    coverage[f] = n_filled / len(records)

# Plausibility: на значениях НЕ-`нет` — % прошедших чек
plausibility = {}
for f in EVAL_FIELDS:
    if f not in PLAUSIBLE: continue
    checks = [PLAUSIBLE[f](r.get(f)) for r in records]
    checks = [c for c in checks if c is not None]
    plausibility[f] = sum(checks) / len(checks) if checks else None

# Soft similarity vs GT на 124 matched
soft_sim = {}
for f in EVAL_FIELDS:
    sims = [char_jaccard(g.get(f), p.get(f)) for g, p in matches]
    soft_sim[f] = sum(sims) / len(sims) if sims else 0.0

# Strict acc уже посчитан в field_stats
def strict_acc(f):
    s = field_stats.get(f)
    return s['correct'] / s['total'] if s and s['total'] else 0.0

# ── Вывод таблицы ──
print(f'{"FIELD":<26} {"COVERAGE":>10} {"PLAUSIBLE":>12} {"SOFT SIM":>10} {"STRICT":>9}')
print('─' * 70)
for f in EVAL_FIELDS:
    cov = f'{coverage[f]:.0%}'
    plaus = f'{plausibility[f]:.0%}' if plausibility.get(f) is not None else '—'
    soft = f'{soft_sim[f]:.2f}'
    strict = f'{strict_acc(f):.0%}'
    print(f'{f:<26} {cov:>10} {plaus:>12} {soft:>10} {strict:>9}')

# Сводные средние
avg_cov = sum(coverage.values()) / len(coverage)
avg_soft = sum(soft_sim.values()) / len(soft_sim)
print('─' * 70)
print(f'{"AVERAGE":<26} {avg_cov:>9.0%}{"":>12} {avg_soft:>10.2f} {sum(strict_acc(f) for f in EVAL_FIELDS)/len(EVAL_FIELDS):>9.0%}')

# Сохраняем в eval_report
report = json.loads(EVAL_REPORT.read_text()) if EVAL_REPORT.exists() else {}
report['coverage']     = coverage
report['plausibility'] = plausibility
report['soft_sim']     = soft_sim
EVAL_REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=1))
print(f'\nЗаписан в {EVAL_REPORT}')

FIELD                        COVERAGE    PLAUSIBLE   SOFT SIM    STRICT
──────────────────────────────────────────────────────────────────────
product_name                      43%            —       0.43        0%
price_default                     38%         100%       0.33        1%
price_card                        66%         100%       0.73       64%
price_discount                     0%            —       1.00      100%
barcode                            5%         100%       0.11        4%
discount_amount                   37%         100%       0.68       68%
id_sku                             6%         100%       0.13        6%
print_datetime                     0%            —       0.34       34%
code                               0%            —       0.25       25%
additional_info                   13%         100%       0.68       61%
color                            100%         100%       0.99       99%
special_symbols                    6%         100%       0.34    